Para identificar os caracteres quebrados e seus bytes, iteramos o arquivo e adicionamos as sequências de bytes e as palavras.

In [0]:
import re
from collections import defaultdict

def printHex(filepath):
    with open(filepath, "rb") as f:
        content = f.read()

    palavras = defaultdict(set)

    tokens = re.split(b'[;\n\r \t]+', content)
    for token in tokens:
        if not token:
            continue
        
        bytes_quebrados = re.findall(b'[\x80-\xff]+', token) # x80 até xff não são ascii

        if bytes_quebrados:
            palavra_formatada = ""
            for b in token:
                if b > 127:
                    palavra_formatada += f"\\x{b:02X}"
                else:
                    palavra_formatada += chr(b)

            chave = tuple(bytes_quebrados)
            palavras[chave].add(palavra_formatada)

    sorted_c = sorted(palavras.keys(), key=lambda k: len(b''.join(k)), reverse=True)

    for chave in sorted_c:
        c = ",".join(["".join([f"\\x{b:02X}" for b in seq]) for seq in chave])
        p = sorted(list(palavras[chave]))
        print(f"HEX: [{c}]")
        print("-" * 40)
        for palavra in p:
            print(f"  {palavra}")
            print("")


printHex("/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/raw/seguros_vida_grande.csv")

HEX: [\xC7\xF5\xC7\x9C]
----------------------------------------
  Concei\xC7\xF5\xC7\x9Co

HEX: [\xC7\x9C]
----------------------------------------
  Arag\xC7\x9Co

  Cau\xC7\x9C

  Jo\xC7\x9Co

  Le\xC7\x9Co

HEX: [\xC7\xF0]
----------------------------------------
  Al\xC7\xF0cia

  Aux\xC7\xF0lio

  Ben\xC7\xF0cio

  Cec\xC7\xF0lia

  Helo\xC7\xF0sa

  L\xC7\xF0via

  La\xC7\xF0s

  Lav\xC7\xF0nia

  Let\xC7\xF0cia

  Lu\xC7\xF0sa

  Ol\xC7\xF0via

  Vin\xC7\xF0cius

HEX: [\xC7\xAD]
----------------------------------------
  B\xC7\xADrbara

  Di\xC7\xADria

  Elo\xC7\xAD

  Goi\xC7\xADs

  Nat\xC7\xADlia

  Ot\xC7\xADvio

  S\xC7\xAD

HEX: [\xC7\xF5]
----------------------------------------
  Doen\xC7\xF5as

  Foga\xC7\xF5a

  Gon\xC7\xF5alves

  Mendon\xC7\xF5a

HEX: [\xC7\xBD]
----------------------------------------
  C\xC7\xBDmara

HEX: [\xC7\xA7]
----------------------------------------
  Ara\xC7\xA7jo

  J\xC7\xA7lia

HEX: [\xC7\x8D]
----------------------------------------
 

Com o resultado, montamos a correspondência entre os byes e os caracteres corretos.

In [0]:
replacement_map = {
    b'\xc7\xf5': "ç".encode('utf-8'),
    b'\xc7\x9c': "ã".encode('utf-8'),
    b'\xc7\xf0': "í".encode('utf-8'),
    b'\xc7\xad': "á".encode('utf-8'),
    b'\xc7\xbd': "â".encode('utf-8'),
    b'\xc7\xa7': "ú".encode('utf-8'),
    b'\xc7\x8d': "Í".encode('utf-8'),
    b'\xc7\xa6': "ê".encode('utf-8'),
    b'\xc7\xfc': "ó".encode('utf-8'),
    b'\xc7\xb8': "é".encode('utf-8'),
    b'\xc7\x81': "Á".encode('utf-8'),
    b'\xc7\x9c': "ã".encode('utf-8'),
    b'\xc7\xef': "ô".encode('utf-8'),
}

def fix_encoding_issues(input_path, output_path, replacement_map):

    with open(input_path, 'rb') as f:
        content = f.read()

    for bad_bytes, good_bytes in replacement_map.items():
        content = content.replace(bad_bytes, good_bytes)
        print(f"{bad_bytes.hex()} => {good_bytes.hex()}")

    try:
        content.decode('utf-8')
    except UnicodeDecodeError as e:
        print(f"\nUTF-8 inválido na posição {e.start}.")

    with open(output_path, 'wb') as f:
        f.write(content)

fix_encoding_issues("/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/raw/seguros_vida_grande.csv",
"/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/processed/seguros_vida_grande_encoding.csv", replacement_map)

c7f5 => c3a7
c79c => c3a3
c7f0 => c3ad
c7ad => c3a1
c7bd => c3a2
c7a7 => c3ba
c78d => c38d
c7a6 => c3aa
c7fc => c3b3
c7b8 => c3a9
c781 => c381
c7ef => c3b4
